# Chase / Wumpus — Opponent diversity & stochasticity

**Ερευνητικό ερώτημα:** Η ποικιλία αντιπάλων και η μεταβολή του `p_fail` κατά την
εκπαίδευση βελτιώνουν τη γενίκευση σε νέες πολιτικές και άγνωστες στρατηγικές;

Αυτοτελής πειραματική επέκταση του **Q-Catch**, των Christos Karatsalos και
Konstantinos Mpouros. Οι χάρτες και η προτεραιότητα κανόνων προέρχονται από το
`QLearning_Chase-main (1).zip`. Ο κώδικας εδώ είναι νέα, ενοποιημένη υλοποίηση·
δεν τροποποιεί το αρχικό ZIP ούτε αναπαράγει αυτούσια τα παλαιά αποτελέσματα.

**Run all:** το προεπιλεγμένο `smoke` εκτελεί σύντομο έλεγχο ολόκληρης της ροής.
Οι εμφανιζόμενες επιδόσεις smoke **δεν είναι ερευνητικά αποτελέσματα**.
Αλλάξτε `PROFILE = "full"` για το πλήρες πρωτόκολλο. Το `pilot` είναι ενδιάμεσο.
Απαιτούνται Python ≥3.10, PyTorch, NumPy, SciPy, pandas και matplotlib.
Δεν χρειάζεται το ZIP, GPU, nashpy ή πρόσβαση σε εξωτερικό dataset.

Περιλαμβάνονται: τέσσερα factorial πειράματα, πολλαπλοί σταθεροί αντίπαλοι ως
controls, παγωμένες train/validation/test συλλογές, ανεξάρτητοι agents ανά ρόλο,
προαιρετική προσαρμοστική δειγματοληψία, CSV/JSON/checkpoints, confidence intervals,
διαφορές ανά seed και γραφήματα. Το πλήρες πείραμα μπορεί να είναι χρονοβόρο.

**Έκδοση με tests (27/09/2026):** Εκτέλεσε το notebook μέσα από το αποσυμπιεσμένο project. Το test suite τρέχει πριν από τα πειράματα. Η εκπαίδευση είναι αρχικά απενεργοποιημένη: όρισε `RUN_EXPERIMENTS=True` όταν θέλεις νέο πείραμα. Τα παλαιά pilot αποτελέσματα δεν έχουν επανυπολογιστεί με τις διορθώσεις.

In [ ]:
# Uncomment ONLY if dependencies are missing (Jupyter / Colab):
# %pip install torch numpy scipy pandas matplotlib
import copy
import hashlib
import json
import math
import os
import platform
import time
import warnings
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict, replace
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy.optimize import linprog
from scipy.stats import t as student_t
import torch
from torch import nn
from IPython.display import display

torch.set_num_threads(1)  # small MLPs: avoid CPU thread overhead
torch.use_deterministic_algorithms(True)
plt.rcParams.update({"figure.dpi": 115, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
print("Python", platform.python_version(), "| PyTorch", torch.__version__)

## 1. Πρωτόκολλο και ρυθμίσεις

| Arm | Αντίπαλος | p_fail |
|---|---|---|
| A | Ένας σταθερός, ξεχωριστό control ανά οικογένεια | 0.10 |
| B | Ομοιόμορφα ανά οικογένεια, μετά ανά checkpoint | 0.10 |
| C | Ίδιος σταθερός αντίπαλος με το A | 0.05 / 0.10 / 0.15 |
| D | Πολλοί αντίπαλοι | 0.05 / 0.10 / 0.15 |
| E (optional) | Προσαρμοστικά βάρη οικογένειας × p_fail | ίδιο σύνολο με D |

Τα E βάρη προκύπτουν μόνο από validation επεισόδια απέναντι στο **train pool**.
Ένα ποσοστό 20% παραμένει ομοιόμορφο. Δεν χρησιμοποιούνται test outcomes.
Τα validation βήματα καταγράφονται χωριστά· το E έχει πρόσθετο κόστος.
Δεν πρόκειται για πλήρες PSRO: δεν λύνουμε μεταπαίγνιο ούτε προσθέτουμε best responses.

Το `full` χρησιμοποιεί 5 seeds, 500 επεισόδια ανά test cell και όλους τους
σταθερούς controls. Κάθε policy έχει ακριβώς το ίδιο budget **πραγματικών training
steps**. Η προετοιμασία του pool είναι πρόσθετο, κοινό κόστος που καταγράφεται.
Τα checkpoints αποθηκεύονται με optimizer/replay/RNG για συνέχιση εκπαίδευσης.
Αλλαγή profile/config/code δημιουργεί διαφορετικό output directory.

In [ ]:
PROFILE = "pilot"               # "smoke", "pilot", "full"
RUN_EXPERIMENTS = False
RUN_ADAPTIVE = True             # optional arm E
HOLDOUT_FAMILY = None           # e.g. "dynaq": excluded from training, evaluated as unseen
REWARD_MODE = "potential"      # "potential" or "outcome"; both strictly zero-sum
LAYOUT_NAME = "small"           # "small", "default", "dangerous"
FIXED_FAMILIES_OVERRIDE = None  # e.g. ("fp", "minimax", "dynaq", "dqn")
RUN_TAG = "tests_v2"                  # change to intentionally start a fresh run
OUTPUT_ROOT = Path("chase_population_runs")

@dataclass(frozen=True)
class Config:
    profile: str
    learner_steps: int
    bank_steps: int
    eval_episodes: int
    seeds: tuple
    bank_replicas: int
    fixed_families: tuple
    hidden: int
    batch: int
    warmup: int
    horizon: int = 30
    gamma: float = 0.95
    lr: float = 5e-4
    replay_size: int = 50_000
    train_every: int = 4
    target_every: int = 200
    planning_steps: int = 5
    train_p: tuple = (0.05, 0.10, 0.15)
    test_p: tuple = (0.0, 0.05, 0.10, 0.15, 0.20, 0.30)
    validation_episodes: int = 8
    role_names: tuple = ("chaser", "runner")

PROFILES = {
    "smoke": Config("smoke", 320, 100, 8, (11, 22), 1, ("fp",), 32, 16, 32,
                    replay_size=1024, planning_steps=1, validation_episodes=2),
    "pilot": Config("pilot", 20_000, 5_000, 100, (11, 22, 33), 1,
                    ("fp", "minimax"), 64, 64, 256),
    "full": Config("full", 200_000, 50_000, 500, (11, 22, 33, 44, 55), 2,
                   ("fp", "minimax", "dynaq", "dqn"), 128, 64, 1000),
}
CFG = PROFILES[PROFILE]
TRAIN_FAMILIES = tuple(f for f in ("fp", "minimax", "dynaq", "dqn")
                       if f != HOLDOUT_FAMILY)
controls = FIXED_FAMILIES_OVERRIDE or CFG.fixed_families
controls = tuple(f for f in controls if f in TRAIN_FAMILIES)
assert controls and HOLDOUT_FAMILY in (None, "fp", "minimax", "dynaq", "dqn")
CFG = replace(CFG, fixed_families=controls)
DEVICE = "cpu"  # fully reproducible default; change deliberately for GPU experiments
if PROFILE == "smoke":
    print("SMOKE ONLY — pipeline validation, not evidence of convergence or superiority.")
print(asdict(CFG))

## 2. Κοινό μοντέλο περιβάλλοντος

Ταυτόχρονες ενέργειες STAY/UP/DOWN/LEFT/RIGHT. Διατηρείται η αρχική σύμβαση:
το UP αυξάνει την πρώτη συντεταγμένη. Capture σημαίνει ίδιο κελί ή ανταλλαγή θέσεων.
Προτεραιότητα: **capture → treasure → hazards → timeout**. Αν πεθάνουν και οι
δύο χωρίς προηγούμενο outcome, κερδίζει ο chaser, όπως στο αρχικό περιβάλλον.

Η κατάσταση περιλαμβάνει **υπολειπόμενο χρόνο και γνωστό p_fail**. Το timeout
είναι πραγματικό terminal outcome (νίκη runner), συνεπώς δεν γίνεται bootstrap.
Διακοπή μόνο λόγω training budget δεν μετατρέπεται σε τεχνητό terminal.

Το αρχικό σύνθετο reward ήταν general-sum. Εδώ το βασικό reward είναι ±10 στον
νικητή/ηττημένο. Προαιρετικά προστίθεται `gamma * Phi(s') - Phi(s)`, με
`Phi_terminal = 0` και αντίθετα potentials ανά παίκτη, ώστε `r_A + r_B = 0`.
Ο ίδιος ακριβώς κανόνας χρησιμοποιείται από environment και FP lookahead.
Αποφεύγουμε μη ελεγχόμενη αλλαγή shaping μεταξύ των πειραμάτων.

In [ ]:
MOVES = ((0, 0), (1, 0), (-1, 0), (0, -1), (0, 1))
ACTION_NAMES = ("STAY", "UP", "DOWN", "LEFT", "RIGHT")

@dataclass(frozen=True)
class Layout:
    size: int
    obstacles: frozenset
    pits: frozenset
    wumpus: tuple
    treasure: tuple

LAYOUTS = {
    "small": Layout(5, frozenset({(1, 1), (2, 1)}), frozenset({(2, 2)}), (3, 3), (1, 4)),
    "default": Layout(7, frozenset({(2, 2), (2, 3), (3, 2), (4, 4), (1, 4)}),
                      frozenset({(3, 4), (5, 1)}), (5, 5), (1, 5)),
    "dangerous": Layout(7, frozenset({(3, 3)}),
                        frozenset({(1, 2), (2, 5), (4, 1), (5, 3)}), (5, 5), (0, 6)),
}

@dataclass(frozen=True)
class State:
    a: tuple
    b: tuple
    remaining: int
    p_fail: float

def stable_seed(*parts):
    payload = json.dumps(parts, sort_keys=True, default=str).encode()
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], "little")

def manhattan(x, y):
    return abs(x[0] - y[0]) + abs(x[1] - y[1])

def random_argmax(values, rng):
    values = np.asarray(values)
    return int(rng.choice(np.flatnonzero(np.isclose(values, values.max(), atol=1e-10, rtol=0.0))))

class World:
    def __init__(self, layout, horizon=30, gamma=0.95, reward_mode="potential"):
        self.layout, self.horizon, self.gamma = layout, horizon, gamma
        self.reward_mode = reward_mode
        assert reward_mode in ("potential", "outcome")
        self.safe = tuple((x, y) for x in range(layout.size) for y in range(layout.size)
                          if (x, y) not in layout.obstacles | layout.pits
                          and (x, y) not in (layout.wumpus, layout.treasure))

    def initial(self, p_fail, rng):
        assert 0 <= p_fail <= 1
        i, j = rng.choice(len(self.safe), 2, replace=False)
        return State(self.safe[i], self.safe[j], self.horizon, float(p_fail))

    def move(self, pos, action, failed):
        if failed:
            return pos
        dx, dy = MOVES[action]
        nxt = (pos[0] + dx, pos[1] + dy)
        if not (0 <= nxt[0] < self.layout.size and 0 <= nxt[1] < self.layout.size):
            return pos
        return pos if nxt in self.layout.obstacles else nxt

    def dead(self, pos):
        return pos in self.layout.pits or pos == self.layout.wumpus

    def potential(self, s):
        scale = 2 * (self.layout.size - 1)
        return (-0.5 * manhattan(s.a, s.b)
                -0.25 * manhattan(s.a, self.layout.treasure)
                +0.25 * manhattan(s.b, self.layout.treasure)) / scale

    def transition(self, s, aa, ab, failed=(False, False)):
        assert s.remaining > 0 and 0 <= aa < 5 and 0 <= ab < 5
        a, b = self.move(s.a, aa, failed[0]), self.move(s.b, ab, failed[1])
        ns = State(a, b, s.remaining - 1, s.p_fail)
        capture = a == b or (a == s.b and b == s.a)
        da, db = self.dead(a), self.dead(b)
        winner, reason = None, "ongoing"
        if capture:
            winner, reason = 0, "capture"
        elif a == self.layout.treasure or b == self.layout.treasure:
            winner, reason = (0 if a == self.layout.treasure else 1), "treasure"
        elif da or db:
            winner = 0 if db else 1
            victim_pos = b if db else a
            reason = "pit" if victim_pos in self.layout.pits else "wumpus"
        elif ns.remaining == 0:
            winner, reason = 1, "timeout"
        done = winner is not None
        base = 0.0 if not done else (10.0 if winner == 0 else -10.0)
        shaping = 0.0
        if self.reward_mode == "potential":
            shaping = self.gamma * (0.0 if done else self.potential(ns)) - self.potential(s)
        r = base + shaping
        info = {"winner": winner, "reason": reason, "dead_a": da, "dead_b": db,
                "base_a": base, "shaping_a": shaping}
        return ns, np.array([r, -r], dtype=np.float64), done, info

    def step(self, s, aa, ab, rng):
        return self.transition(s, aa, ab, tuple(rng.random(2) < s.p_fail))

    @lru_cache(maxsize=20_000)
    def expected_payoff(self, s):
        # Immutable per-state joint payoff matrix; cache does not change policy.
        p = s.p_fail
        failures = [((False, False), (1-p)**2), ((True, False), p*(1-p)),
                    ((False, True), (1-p)*p), ((True, True), p*p)]
        q = np.zeros((5, 5))
        for aa in range(5):
            for ab in range(5):
                q[aa, ab] = sum(prob * self.transition(s, aa, ab, fail)[1][0]
                               for fail, prob in failures if prob > 0)
        q.setflags(write=False)
        return q

WORLD = World(LAYOUTS[LAYOUT_NAME], CFG.horizon, CFG.gamma, REWARD_MODE)

def features(world, s, role):
    # Fixed A/B coordinate convention plus explicit role. No incomplete state flip.
    d = world.layout.size - 1
    a, b, tr, wu = s.a, s.b, world.layout.treasure, world.layout.wumpus
    nearby = lambda pos, cells: float(any(manhattan(pos, cell) == 1 for cell in cells))
    return np.array([a[0]/d, a[1]/d, b[0]/d, b[1]/d,
                     manhattan(a, b)/(2*d), manhattan(a, tr)/(2*d),
                     manhattan(b, tr)/(2*d), manhattan(a, wu)/(2*d),
                     manhattan(b, wu)/(2*d), nearby(a, world.layout.pits),
                     nearby(b, world.layout.pits), float(manhattan(a, wu) == 1),
                     float(manhattan(b, wu) == 1), s.remaining/world.horizon,
                     s.p_fail, float(role)], dtype=np.float32)

## 3. Αντίπαλοι και συμβάσεις

* FP: empirical opponent actions + κοινό one-step expected reward. Οι beliefs
  εκπαιδεύονται κατά το bootstrap και παραμένουν σταθερές σε learner training/test.
* Minimax-Q: sparse πίνακας `Q[state][own_action, opponent_action]` για κάθε ρόλο.
  Κοινός LP solver για μικτές στρατηγικές και Bellman values.
* Stochastic Dyna-Minimax-Q: **ίδιος** κανόνας αξίας με Minimax-Q + εμπειρικό
  πιθανοτικό μοντέλο joint-action transitions. Αυτή είναι σαφώς ονομασμένη
  παραλλαγή Dyna, όχι το pure-action maximin της αρχικής υλοποίησης.
* Dueling Double DQN: ανεξάρτητο δίκτυο ανά ρόλο, replay, Huber loss, target net.

Όλοι γνωρίζουν το `p_fail`. Το FP χρησιμοποιεί και ακριβές one-step μοντέλο·
αυτή η πληροφοριακή διαφορά καταγράφεται και δεν αποτελεί ανεξάρτητη σύγκριση
sample efficiency των οικογενειών. Κύριος learner σε όλα τα arms είναι το ίδιο DQN.
Δεν χρησιμοποιείται action masking: αποτυχίες κίνησης/τοίχοι παραμένουν μέρος του task.
Οι tabular αντίπαλοι χρησιμοποιούν bins p_fail στις τιμές εκπαίδευσης (nearest
bin, clipping στα άκρα). Έτσι μια νέα τιμή p_fail δεν μηδενίζει αυτόματα όλη την
πολιτική τους. Πρόκειται για ρητή προσέγγιση· το DQN λαμβάνει την πραγματική τιμή.

In [ ]:
LP_FAILURES = 0
def maximin(q):
    global LP_FAILURES
    n, m = q.shape
    if np.ptp(q) < 1e-12:
        return np.full(n, 1/n), float(q.flat[0])
    res = linprog(np.r_[np.zeros(n), -1.0],
                  A_ub=np.c_[-q.T, np.ones(m)], b_ub=np.zeros(m),
                  A_eq=np.array([np.r_[np.ones(n), 0.0]]), b_eq=[1.0],
                  bounds=[(0, 1)]*n + [(None, None)], method="highs")
    if not res.success:
        LP_FAILURES += 1
        raise RuntimeError("Minimax LP failed: " + res.message)
    pi = np.maximum(res.x[:n], 0)
    pi /= pi.sum()
    return pi, float(np.min(pi @ q))

class Scripted:
    family = "scripted"
    def __init__(self, role, kind="random"):
        self.role, self.kind = role, kind

    def act(self, world, s, rng, epsilon=0.0):
        if self.kind == "random" or rng.random() < epsilon:
            return int(rng.integers(5))
        own, other = (s.a, s.b) if self.role == 0 else (s.b, s.a)
        values = []
        for act in range(5):
            nxt = world.move(own, act, False)
            dist = manhattan(nxt, other)
            treasure = manhattan(nxt, world.layout.treasure)
            risk = 100.0 * world.dead(nxt)
            if self.kind == "safe":  # bootstrap sparring partner only
                value = (-dist if self.role == 0 else dist) - 0.3*treasure - risk
            elif self.kind == "treasure":  # withheld family
                value = -2*treasure + (-0.2*dist if self.role == 0 else 0.2*dist) - risk
            elif self.kind == "ambush":  # withheld family: goals change with time
                if self.role == 0:
                    target = world.layout.treasure if dist > 2 else other
                    value = -manhattan(nxt, target) - risk
                else:
                    value = (dist if s.remaining < world.horizon/2 else -treasure) - risk
            else:
                raise ValueError(self.kind)
            values.append(value)
        return random_argmax(values, rng)

class FP:
    family = "fp"
    def __init__(self, role, seed=0):
        self.role = role
        self.counts = {}
    def belief_key(self, s):
        # Pool time/p_fail for beliefs only; reward lookahead remains time/p aware.
        return s.a + s.b
    def act(self, world, s, rng, epsilon=0.0):
        if rng.random() < epsilon:
            return int(rng.integers(5))
        counts = self.counts.get(self.belief_key(s), np.ones(5))
        belief = counts / counts.sum()
        payoff = world.expected_payoff(s)
        own_payoff = payoff if self.role == 0 else -payoff.T
        return random_argmax(own_payoff @ belief, rng)
    def learn(self, world, s, aa, ab, rewards, ns, done):
        key = self.belief_key(s)
        self.counts.setdefault(key, np.ones(5))[ab if self.role == 0 else aa] += 1

class Minimax:
    family = "minimax"
    def __init__(self, role, seed=0, gamma=0.95, alpha=0.15):
        self.role, self.gamma, self.alpha = role, gamma, alpha
        self.q, self.cache = {}, {}
        self.rng = np.random.default_rng(seed)
    def table(self, s):
        return self.q.get(self.key(s), np.zeros((5, 5)))
    def key(self, s):
        p = min(CFG.train_p, key=lambda value: abs(value-s.p_fail))
        return replace(s, p_fail=p)
    def policy(self, s):
        s = self.key(s)
        if s not in self.q:
            return np.full(5, 0.2), 0.0
        if s not in self.cache:
            self.cache[s] = maximin(self.q[s])
        return self.cache[s]
    def act(self, world, s, rng, epsilon=0.0):
        if rng.random() < epsilon:
            return int(rng.integers(5))
        return int(rng.choice(5, p=self.policy(s)[0]))
    def update_pair(self, s, own, opp, reward, ns, done):
        target = reward if done else reward + self.gamma*self.policy(ns)[1]
        s = self.key(s)
        q = self.q.setdefault(s, np.zeros((5, 5)))
        q[own, opp] += self.alpha*(target-q[own, opp])
        self.cache.pop(s, None)
    def learn(self, world, s, aa, ab, rewards, ns, done):
        own, opp = (aa, ab) if self.role == 0 else (ab, aa)
        self.update_pair(s, own, opp, float(rewards[self.role]), ns, done)

class DynaMinimax(Minimax):
    family = "dynaq"
    def __init__(self, role, seed=0, gamma=0.95, planning=5):
        super().__init__(role, seed, gamma)
        self.planning = planning
        self.model, self.model_keys = {}, []
        self.planning_updates = 0
    def learn(self, world, s, aa, ab, rewards, ns, done):
        own, opp = (aa, ab) if self.role == 0 else (ab, aa)
        reward = float(rewards[self.role])
        self.update_pair(s, own, opp, reward, ns, done)
        key = (s, own, opp)
        if key not in self.model:
            self.model[key] = Counter()
            self.model_keys.append(key)
        # Store joint (next state, reward, terminal) outcomes and empirical counts.
        self.model[key][(ns, reward, bool(done))] += 1
        for _ in range(self.planning):
            k = self.model_keys[int(self.rng.integers(len(self.model_keys)))]
            outcomes = self.model[k]
            choices, counts = list(outcomes), np.array(list(outcomes.values()), dtype=float)
            nxt, rew, terminal = choices[int(self.rng.choice(len(choices), p=counts/counts.sum()))]
            self.update_pair(*k, rew, nxt, terminal)
            self.planning_updates += 1

In [ ]:
class DuelingNetwork(nn.Module):
    def __init__(self, inputs=16, hidden=64):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(inputs, hidden), nn.ReLU(),
                                  nn.Linear(hidden, hidden), nn.ReLU())
        self.value = nn.Linear(hidden, 1)
        self.advantage = nn.Linear(hidden, 5)
    def forward(self, x):
        h = self.body(x)
        a = self.advantage(h)
        return self.value(h) + a - a.mean(dim=-1, keepdim=True)

class Replay:
    def __init__(self, capacity, dim=16):
        self.capacity, self.pos, self.size = capacity, 0, 0
        self.x = np.zeros((capacity, dim), np.float32)
        self.nx = np.zeros_like(self.x)
        self.a = np.zeros(capacity, np.int64)
        self.r = np.zeros(capacity, np.float32)
        self.done = np.zeros(capacity, np.float32)
    def push(self, x, a, r, nx, done):
        i = self.pos
        self.x[i], self.nx[i], self.a[i], self.r[i], self.done[i] = x, nx, a, r, done
        self.pos = (i+1) % self.capacity
        self.size = min(self.size+1, self.capacity)
    def sample(self, n, rng, device):
        idx = rng.choice(self.size, n, replace=False)
        return [torch.as_tensor(arr[idx], device=device)
                for arr in (self.x, self.a, self.r, self.nx, self.done)]

class DQN:
    family = "dqn"
    def __init__(self, role, seed, cfg=None):
        cfg = CFG if cfg is None else cfg
        self.role, self.cfg, self.steps, self.updates = role, cfg, 0, 0
        self.rng = np.random.default_rng(seed)
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(seed)
            self.online = DuelingNetwork(hidden=cfg.hidden).to(DEVICE)
        self.target = copy.deepcopy(self.online).eval()
        self.optimizer = torch.optim.Adam(self.online.parameters(), lr=cfg.lr)
        self.replay = Replay(cfg.replay_size)
        self.losses = []
    def act(self, world, s, rng, epsilon=0.0):
        if rng.random() < epsilon:
            return int(rng.integers(5))
        with torch.no_grad():
            q = self.online(torch.as_tensor(features(world, s, self.role), device=DEVICE)[None])
        return random_argmax(q[0].cpu().numpy(), rng)
    def learn(self, world, s, aa, ab, rewards, ns, done):
        action = aa if self.role == 0 else ab
        self.replay.push(features(world, s, self.role), action, rewards[self.role],
                         features(world, ns, self.role), done)
        self.steps += 1
        if self.replay.size >= max(self.cfg.warmup, self.cfg.batch) and self.steps % self.cfg.train_every == 0:
            x, a, r, nx, terminal = self.replay.sample(self.cfg.batch, self.rng, DEVICE)
            predicted = self.online(x).gather(1, a[:, None]).squeeze(1)
            with torch.no_grad():
                actions = self.online(nx).argmax(dim=1, keepdim=True)
                next_value = self.target(nx).gather(1, actions).squeeze(1)
                target = r + self.cfg.gamma*(1-terminal)*next_value
            loss = nn.functional.smooth_l1_loss(predicted, target)
            self.optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(self.online.parameters(), 10.0)
            self.optimizer.step()
            self.losses.append(float(loss.detach()))
            self.updates += 1
        if self.steps % self.cfg.target_every == 0:
            self.target.load_state_dict(self.online.state_dict())

class FrozenDQN:
    family = "dqn"
    def __init__(self, source):
        self.role = source.role
        self.online = copy.deepcopy(source.online).eval()
        for p in self.online.parameters():
            p.requires_grad_(False)
    act = DQN.act

def frozen_copy(agent):
    if isinstance(agent, DQN):
        return FrozenDQN(agent)
    out = copy.deepcopy(agent)
    if isinstance(out, DynaMinimax):
        out.model, out.model_keys = {}, []  # not used by frozen policies
    return out

def epsilon_at(step, budget):
    return 1.0 + (0.05-1.0)*min(step/max(1, 0.7*budget), 1.0)

def make_agent(family, role, seed):
    if family == "fp": return FP(role, seed)
    if family == "minimax": return Minimax(role, seed, CFG.gamma)
    if family == "dynaq": return DynaMinimax(role, seed, CFG.gamma, CFG.planning_steps)
    if family == "dqn": return DQN(role, seed)
    raise ValueError(family)

def joint_actions(role, own, other):
    return (own, other) if role == 0 else (other, own)

## 4. Ανεξάρτητες συλλογές αντιπάλων και checkpoints

Bootstrap σε random και safe scripted sparring partners, με διαφορετικά seeds
ανά split. Είναι ελεγχόμενα baselines, όχι προϋπάρχοντες βέλτιστοι agents.
Οι learned αντίπαλοι ελέγχονται σε validation πριν ερμηνεύσουμε τη γενίκευση.
Μεγαλύτερο budget μπορεί να απαιτείται, ιδίως για time-indexed tabular agents.

Το train pool περιλαμβάνει final πολιτικές FP/Minimax/Dyna και δύο DQN snapshots.
Validation/test χρησιμοποιούν ανεξάρτητες εκπαιδεύσεις και μόνο final checkpoints.
Οι withheld scripted families `treasure` και `ambush` δεν χρησιμοποιούνται στο
bootstrap ούτε στην εκπαίδευση του κύριου learner. Το holdout αποκλείει ολόκληρη
learned οικογένεια από το learner training pool. Για leave-one-family-out αλλάξτε
`HOLDOUT_FAMILY` και ξανατρέξτε· το output namespace αλλάζει αυτόματα.

Save/load χρησιμοποιεί μόνο checkpoints που παράγει αυτό το notebook. Μην
φορτώνετε μη έμπιστα `.pt` αρχεία με `weights_only=False`.

In [ ]:
CODE_SIGNATURE = "tests-v2-20260927"
RUN_SPEC = {"config": asdict(CFG), "layout": LAYOUT_NAME, "reward": REWARD_MODE,
            "adaptive": RUN_ADAPTIVE, "holdout": HOLDOUT_FAMILY, "tag": RUN_TAG,
            "device": DEVICE, "code": CODE_SIGNATURE,
            "versions": {"torch": str(torch.__version__), "numpy": np.__version__,
                         "scipy": scipy.__version__, "pandas": pd.__version__}}
RUN_ID = hashlib.sha256(json.dumps(RUN_SPEC, sort_keys=True).encode()).hexdigest()[:12]
OUT = OUTPUT_ROOT / f"{PROFILE}_{RUN_ID}"
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "manifest.json").write_text(json.dumps(RUN_SPEC, indent=2), encoding="utf-8")

def atomic_save(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def trusted_load(path):
    return torch.load(path, map_location=DEVICE, weights_only=False)

@dataclass
class PolicyEntry:
    identifier: str
    split: str
    family: str
    role: int
    seed: int
    checkpoint: str
    agent: object

def parameter_digest(agent):
    """Learned-state hash (excludes policy caches and random-number states)."""
    h = hashlib.sha256()
    if hasattr(agent, "online"):
        for k, v in agent.online.state_dict().items():
            h.update(k.encode()); h.update(v.detach().cpu().numpy().tobytes())
    elif hasattr(agent, "q"):
        for k in sorted(agent.q, key=repr):
            h.update(repr(k).encode()); h.update(agent.q[k].tobytes())
    elif hasattr(agent, "counts"):
        for k in sorted(agent.counts, key=repr):
            h.update(repr(k).encode()); h.update(agent.counts[k].tobytes())
    else:
        h.update(repr((agent.role, agent.kind)).encode())
    return h.hexdigest()

def bootstrap(family, role, seed, steps):
    start = time.perf_counter()
    agent = make_agent(family, role, seed)
    env_rng = np.random.default_rng(stable_seed(seed, "bank_environment"))
    action_rng = np.random.default_rng(stable_seed(seed, "bank_actions"))
    opp_rng = np.random.default_rng(stable_seed(seed, "bank_opponent"))
    schedule_rng = np.random.default_rng(stable_seed(seed, "bank_schedule"))
    s, snapshots, episode = None, {}, 0
    for step in range(steps):
        if s is None:
            p = float(schedule_rng.choice(CFG.train_p))
            s = WORLD.initial(p, env_rng)
            opponent = Scripted(1-role, "random" if episode % 2 == 0 else "safe")
            episode += 1
        own = agent.act(WORLD, s, action_rng, epsilon_at(step, steps))
        other = opponent.act(WORLD, s, opp_rng)
        aa, ab = joint_actions(role, own, other)
        ns, rewards, done, _ = WORLD.step(s, aa, ab, env_rng)
        agent.learn(WORLD, s, aa, ab, rewards, ns, done)
        if family == "dqn" and step+1 == max(1, steps//2):
            snapshots["half"] = frozen_copy(agent)
        s = None if done else ns
    snapshots["final"] = frozen_copy(agent)
    stats = {"seconds": time.perf_counter()-start, "steps": steps,
             "gradient_updates": getattr(agent, "updates", 0),
             "planning_updates": getattr(agent, "planning_updates", 0)}
    return snapshots, stats

def build_bank():
    entries, records = [], []
    for split in ("train", "validation", "test"):
        for role in (0, 1):
            for family in ("fp", "minimax", "dynaq", "dqn"):
                for rep in range(CFG.bank_replicas):
                    seed = stable_seed("bank", split, role, family, rep)
                    path = OUT / f"bank_{split}_{role}_{family}_{rep}.pt"
                    if path.exists():
                        payload = trusted_load(path)
                    else:
                        snapshots, stats = bootstrap(family, role, seed, CFG.bank_steps)
                        payload = {"snapshots": snapshots, "stats": stats, "seed": seed}
                        atomic_save(payload, path)
                    for ckpt, agent in payload["snapshots"].items():
                        if split != "train" and ckpt != "final":
                            continue
                        pid = f"{split}_{role}_{family}_{rep}_{ckpt}"
                        entries.append(PolicyEntry(pid, split, family, role, seed, ckpt, agent))
                    records.append({"split": split, "role": role, "family": family,
                                    "replica": rep, **payload["stats"]})
        print(f"Opponent bank: {split} ready", flush=True)
    for role in (0, 1):
        for kind in ("treasure", "ambush"):
            entries.append(PolicyEntry(f"test_{role}_{kind}", "test", kind, role,
                                       stable_seed("withheld", role, kind), "scripted",
                                       Scripted(role, kind)))
    return entries, pd.DataFrame(records)

def training_pool(bank, opponent_role):
    return [e for e in bank if e.split == "train" and e.role == opponent_role
            and e.family in TRAIN_FAMILIES]

## 5. Συνθήκες, sampler και εκπαίδευση

Οι παγωμένοι αντίπαλοι δεν εκτελούν `learn/observe` κατά το learner training.
Για ένα fixed control χρησιμοποιείται ακριβώς η ίδια πολιτική στα A και C,
με κυκλική επιλογή replica ανά training seed (final checkpoint).
Το DQN epsilon μειώνεται βάσει **environment steps**, όχι επεισοδίων.

Το adaptive arm ενημερώνει EMA δυσκολίας από ανεξάρτητα validation επεισόδια.
Αξιολογεί οικογένεια × p_fail με τυχαίο checkpoint της οικογένειας.
Ο learner δεν μαθαίνει από validation transitions και το τελικό checkpoint
επιλέγεται βάσει προκαθορισμένου budget, όχι test/validation score.

In [ ]:
@dataclass(frozen=True)
class Condition:
    name: str
    arm: str
    fixed_family: str | None = None
    randomized_p: bool = False
    adaptive: bool = False

CONDITIONS = []
for family in CFG.fixed_families:
    CONDITIONS += [Condition(f"A_{family}", "A", family),
                   Condition(f"C_{family}", "C", family, True)]
CONDITIONS += [Condition("B_pool", "B"), Condition("D_pool_random_p", "D", randomized_p=True)]
if RUN_ADAPTIVE:
    CONDITIONS.append(Condition("E_adaptive", "E", randomized_p=True, adaptive=True))
display(pd.DataFrame([asdict(c) for c in CONDITIONS]))
print("Learner runs:", len(CONDITIONS)*len(CFG.seeds)*2,
      "| Total real learner training steps:", f"{len(CONDITIONS)*len(CFG.seeds)*2*CFG.learner_steps:,}")

def rollout(agent, opponent, role, p, scenario_seed, stream="eval"):
    # Reset and failure randomness independent from both policies; paired across arms.
    env_rng = np.random.default_rng(stable_seed(scenario_seed, stream, "env"))
    own_rng = np.random.default_rng(stable_seed(scenario_seed, stream, "own"))
    opp_rng = np.random.default_rng(stable_seed(scenario_seed, stream, "opp"))
    s = WORLD.initial(p, env_rng)
    total, discounted, length = 0.0, 0.0, 0
    while True:
        aa, ab = joint_actions(role, agent.act(WORLD, s, own_rng), opponent.act(WORLD, s, opp_rng))
        ns, rewards, done, info = WORLD.step(s, aa, ab, env_rng)
        total += float(rewards[role])
        discounted += CFG.gamma**length*float(rewards[role])
        length += 1
        if done:
            return {"win": int(info["winner"] == role), "steps": length,
                    "return": total, "discounted_return": discounted,
                    "reason": info["reason"],
                    "own_hazard": int(info["dead_a"] if role == 0 else info["dead_b"]),
                    "capture_steps": float(length) if info["reason"] == "capture" else np.nan}
        s = ns

class Sampler:
    def __init__(self, pool, condition, seed):
        self.condition = condition
        self.rng = np.random.default_rng(stable_seed(seed, "sampler"))
        self.by_family = {}
        for e in pool:
            self.by_family.setdefault(e.family, []).append(e)
        self.families = sorted(self.by_family)
        self.ps = CFG.train_p if condition.randomized_p else (0.10,)
        if condition.fixed_family:
            candidates = [e for e in self.by_family[condition.fixed_family] if e.checkpoint == "final"]
            self.fixed = sorted(candidates, key=lambda e: e.identifier)[CFG.seeds.index(seed) % len(candidates)]
            self.families = [condition.fixed_family]
        self.cells = [(f, p) for f in self.families for p in self.ps]
        self.difficulty = np.full(len(self.cells), 0.5)
        self.history = []

    def probabilities(self):
        if not self.condition.adaptive:
            return np.full(len(self.cells), 1/len(self.cells))
        hard = (self.difficulty + 0.05)**2
        return 0.2/len(hard) + 0.8*hard/hard.sum()

    def sample(self):
        index = int(self.rng.choice(len(self.cells), p=self.probabilities()))
        family, p = self.cells[index]
        if self.condition.fixed_family:
            return self.fixed, p
        entries = self.by_family[family]
        return entries[int(self.rng.integers(len(entries)))], p

    def calibrate(self, agent, role, seed, checkpoint):
        spent = 0
        for i, (family, p) in enumerate(self.cells):
            entries = self.by_family[family]
            wins = []
            for ep in range(CFG.validation_episodes):
                vs = stable_seed("adaptive_validation", seed, role, checkpoint, family, p, ep)
                chooser = np.random.default_rng(vs)
                entry = entries[int(chooser.integers(len(entries)))]
                result = rollout(agent, entry.agent, role, p, vs, "validation")
                wins.append(result["win"]); spent += result["steps"]
            self.difficulty[i] = 0.5*self.difficulty[i] + 0.5*(1-np.mean(wins))
        for (f, p), probability in zip(self.cells, self.probabilities()):
            self.history.append({"step": checkpoint, "family": f, "p_fail": p,
                                 "probability": float(probability)})
        return spent

def train_condition(bank, role, seed, condition):
    path = OUT / f"learner_{role}_{seed}_{condition.name}.pt"
    if path.exists():
        state = trusted_load(path)
        if state["complete"]:
            return state
    else:
        state = {"agent": DQN(role, stable_seed("learner_init", role, seed)),
                 "sampler": Sampler(training_pool(bank, 1-role), condition, seed),
                 "env_rng": np.random.default_rng(stable_seed(seed, role, "train_env")),
                 "own_rng": np.random.default_rng(stable_seed(seed, role, "train_own")),
                 "opp_rng": np.random.default_rng(stable_seed(seed, role, "train_opp")),
                 "step": 0, "s": None, "entry": None, "episode": 0, "ep_return": 0.0,
                 "ep_steps": 0, "logs": [], "validation_steps": 0, "seconds": 0.0,
                 "complete": False}
    learner, sampler = state["agent"], state["sampler"]
    interval = max(1, CFG.learner_steps//4)
    start = time.perf_counter()
    before = {e.identifier: parameter_digest(e.agent) for e in training_pool(bank, 1-role)}
    for step in range(state["step"], CFG.learner_steps):
        if state["s"] is None:
            entry, p = sampler.sample()
            state["entry"] = entry
            state["s"] = WORLD.initial(p, state["env_rng"])
            state["episode"] += 1
            state["ep_return"], state["ep_steps"] = 0.0, 0
        s, entry = state["s"], state["entry"]
        own = learner.act(WORLD, s, state["own_rng"], epsilon_at(step, CFG.learner_steps))
        other = entry.agent.act(WORLD, s, state["opp_rng"])
        aa, ab = joint_actions(role, own, other)
        ns, rewards, done, info = WORLD.step(s, aa, ab, state["env_rng"])
        learner.learn(WORLD, s, aa, ab, rewards, ns, done)
        state["ep_return"] += float(rewards[role]); state["ep_steps"] += 1
        if done:
            state["logs"].append({"step": step+1, "episode": state["episode"],
                                  "win": int(info["winner"] == role), "reason": info["reason"],
                                  "return": state["ep_return"], "steps": state["ep_steps"],
                                  "opponent": entry.identifier, "family": entry.family,
                                  "p_fail": s.p_fail})
        state["s"] = None if done else ns
        state["step"] = step+1
        if (step+1) % interval == 0:
            if condition.adaptive and step+1 < CFG.learner_steps:
                state["validation_steps"] += sampler.calibrate(learner, role, seed, step+1)
            state["seconds"] += time.perf_counter()-start
            atomic_save(state, path)
            start = time.perf_counter()
    state["seconds"] += time.perf_counter()-start
    state["complete"] = True
    assert learner.steps == CFG.learner_steps
    assert before == {e.identifier: parameter_digest(e.agent) for e in training_pool(bank, 1-role)}
    atomic_save(state, path)
    return state

## Αυτοματοποιημένοι έλεγχοι

Tests των πραγματικών code cells και των αρχικών Python modules. Περιλαμβάνουν σύντομες πραγματικές εκπαιδεύσεις και επανέναρξη από checkpoint. Αποτυχία test διακόπτει την εκτέλεση.

In [ ]:
# Execute the independent tests before spending time on training.
# Open this notebook inside the extracted QCatch folder (or its notebooks/ subfolder).
import subprocess
import sys
from pathlib import Path
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "tests" / "test_pilot_environment.py").exists()), None)
if project_root is None:
    raise FileNotFoundError("Extract the complete QCatch package and open its notebook. See TESTING.md.")
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=project_root, check=True)


## 6. Έλεγχοι ορθότητας πριν την εκπαίδευση

Οι έλεγχοι καλύπτουν τα ουσιαστικά προβλήματα της αρχικής ανάλυσης: pits και
κοινά terminal rules, άξονες runner Q, χρονικό όριο, zero-sum reward, stochastic
transitions, μικτές στρατηγικές, frozen policies και διατήρηση RNG/checkpoints.

In [ ]:
def correctness_checks():
    checked = []
    # A runner's own actions are rows; it must select row 1, not transpose again.
    mm = Minimax(role=1)
    s = WORLD.initial(0.10, np.random.default_rng(1))
    mm.q[s] = np.array([[0]*5, [10]*5, [0]*5, [0]*5, [0]*5], float)
    assert mm.policy(s)[0][1] > 0.999
    assert mm.act(WORLD, s, np.random.default_rng(2)) == 1
    pi, value = maximin(np.array([[1., -1.], [-1., 1.]]))
    assert np.allclose(pi, [.5, .5]) and abs(value) < 1e-8
    checked.append("runner own-action axes + mixed minimax")

    layout = Layout(3, frozenset(), frozenset({(1, 1)}), (2, 0), (2, 2))
    w = World(layout, horizon=5)
    pit = State((1, 0), (0, 2), 5, 0.0)
    ns, reward, done, info = w.transition(pit, 4, 0)
    assert done and info["winner"] == 1 and info["reason"] == "pit"
    assert np.isclose(w.expected_payoff(pit)[4, 0], reward[0])
    treasure_vs_pit = State((2, 1), (1, 0), 5, 0.0)
    _, _, _, info = w.transition(treasure_vs_pit, 4, 4)
    assert info["winner"] == 0 and info["reason"] == "treasure"
    swap = State((0, 0), (0, 1), 5, 0.0)
    assert w.transition(swap, 4, 3)[3]["reason"] == "capture"
    checked.append("pit / treasure priority / cross-path capture / FP shared model")

    early = State((0, 0), (0, 2), 5, 0.1)
    last = replace(early, remaining=1)
    assert not w.transition(early, 0, 0)[2]
    assert w.transition(last, 0, 0)[3]["winner"] == 1
    assert not np.array_equal(features(w, early, 0), features(w, last, 0))
    assert not np.array_equal(features(w, early, 0), features(w, replace(early, p_fail=.3), 0))
    checked.append("time/p_fail observed + timeout truly terminal")

    rng = np.random.default_rng(4)
    for _ in range(100):
        st = WORLD.initial(float(rng.choice([0, .1, .3, 1])), rng)
        _, r, _, _ = WORLD.step(st, int(rng.integers(5)), int(rng.integers(5)), rng)
        assert abs(r.sum()) < 1e-10
    st = State((0, 0), (0, 2), 5, 1.0)
    nxt, *_ = w.step(st, 1, 1, rng)
    assert nxt.a == st.a and nxt.b == st.b
    checked.append("zero-sum and p_fail=1")

    net = DQN(0, 123)
    frozen = frozen_copy(net)
    before = parameter_digest(frozen)
    for _ in range(max(CFG.warmup, CFG.batch)+8):
        st = WORLD.initial(.1, rng)
        nx, r, terminal, _ = WORLD.step(st, 0, 0, rng)
        net.learn(WORLD, st, 0, 0, r, nx, terminal)
    assert net.updates > 0 and parameter_digest(frozen) == before
    assert all(np.isfinite(net.losses))
    save_path = OUT / "checkpoint_roundtrip_test.pt"
    atomic_save(net, save_path)
    loaded = trusted_load(save_path)
    assert parameter_digest(net) == parameter_digest(loaded)
    # Continue both from identical replay + RNG states: exact next updates on CPU.
    for _ in range(CFG.train_every):
        net.learn(WORLD, st, 0, 0, r, nx, terminal)
        loaded.learn(WORLD, st, 0, 0, r, nx, terminal)
    assert parameter_digest(net) == parameter_digest(loaded)
    save_path.unlink()
    checked.append("DQN learns / frozen snapshot / exact checkpoint continuation")
    return checked

CHECKS = correctness_checks()
for message in CHECKS:
    print("PASS:", message)

## 7. Εκτέλεση και κοινό test set

Οι τρεις κατηγορίες αξιολόγησης είναι:
* `seen_policy`: ακριβείς final αντίπαλοι από το train pool.
* `unseen_policy`: ανεξάρτητο test seed γνωστής οικογένειας.
* `unseen_family`: withheld scripted families και τυχόν `HOLDOUT_FAMILY`.

Το "seen" αναφέρεται στο κοινό διαθέσιμο training pool. Ένας fixed learner έχει
πράγματι αντιμετωπίσει μόνο ένα μέλος του. Καταγράφεται και ξεχωριστό
`actually_seen` flag ανά policy/condition, ώστε να αποφεύγεται η σύγχυση.
Για κύρια σύγκριση χρησιμοποιούμε **ίδιο test set για όλα τα arms**.

Κάθε scenario seed εξαρτάται από role/opponent/p_fail/episode και **όχι από arm
ή training seed**. Έτσι όλοι βλέπουν ίδιες αρχικές θέσεις και ίδιες ακολουθίες
τυχαίων αποτυχιών. Χωριστά RNG streams για τους δύο παίκτες. Τα test episodes
δεν ενημερώνουν weights, beliefs, replay ή sampler. Τα LP caches επιτρέπονται.

In [ ]:
def evaluation_entries(bank, opponent_role):
    train_final = [e for e in training_pool(bank, opponent_role) if e.checkpoint == "final"]
    test = [e for e in bank if e.role == opponent_role and e.split == "test"]
    return train_final + test

def opponent_group(entry):
    if entry.split == "train": return "seen_policy"
    return "unseen_policy" if entry.family in TRAIN_FAMILIES else "unseen_family"

def evaluate_condition(bank, role, seed, condition, training):
    path = OUT / f"evaluation_{role}_{seed}_{condition.name}.csv"
    entries = evaluation_entries(bank, 1-role)
    expected_rows = len(entries)*len(CFG.test_p)*CFG.eval_episodes
    learner = training["agent"]
    signature_payload = {"code": CODE_SIGNATURE, "run": RUN_SPEC,
                         "condition": asdict(condition), "role": role, "seed": seed,
                         "learner": parameter_digest(learner),
                         "test_p": CFG.test_p, "episodes": CFG.eval_episodes,
                         "fixed_opponent": (training["sampler"].fixed.identifier
                                            if condition.fixed_family else None),
                         "opponents": [(e.identifier, e.family, e.split,
                                        parameter_digest(e.agent)) for e in entries]}
    signature = hashlib.sha256(json.dumps(signature_payload, sort_keys=True).encode()).hexdigest()
    metadata_path = path.with_suffix(".json")
    if path.exists() and metadata_path.exists():
        try:
            metadata = json.loads(metadata_path.read_text())
            checksum = hashlib.sha256(path.read_bytes()).hexdigest()
            if metadata.get("signature") == signature and metadata.get("csv_sha256") == checksum:
                result = pd.read_csv(path)
                if len(result) == expected_rows:
                    return result
        except (ValueError, OSError, pd.errors.ParserError):
            pass  # incomplete or stale cache: regenerate from the current policies

    before = parameter_digest(learner)
    before_opponents = {e.identifier: parameter_digest(e.agent) for e in entries}
    records = []
    start = time.perf_counter()
    for entry in entries:
        for p in CFG.test_p:
            for episode in range(CFG.eval_episodes):
                scenario = stable_seed("final_test", role, entry.identifier, p, episode)
                row = rollout(learner, entry.agent, role, p, scenario)
                if condition.fixed_family:
                    actually_seen = entry.identifier == training["sampler"].fixed.identifier
                else:
                    actually_seen = entry.split == "train" and entry.family in TRAIN_FAMILIES
                records.append({"condition": condition.name, "arm": condition.arm,
                                "fixed_family": condition.fixed_family or "pool", "seed": seed,
                                "role": CFG.role_names[role], "opponent": entry.identifier,
                                "family": entry.family, "group": opponent_group(entry),
                                "actually_seen": actually_seen, "p_fail": p,
                                "outside_train_range": p < min(CFG.train_p) or p > max(CFG.train_p),
                                "high_noise_ood": p > max(CFG.train_p),
                                "episode": episode, "scenario_seed": scenario, **row})
    assert parameter_digest(learner) == before
    assert before_opponents == {e.identifier: parameter_digest(e.agent) for e in entries}
    frame = pd.DataFrame(records)
    temp = path.with_suffix(".csv.tmp")
    frame.to_csv(temp, index=False); os.replace(temp, path)
    metadata = {"evaluation_seconds": time.perf_counter()-start, "episodes": len(frame),
                "signature": signature, "csv_sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
    metadata_tmp = metadata_path.with_suffix(".json.tmp")
    metadata_tmp.write_text(json.dumps(metadata), encoding="utf-8")
    os.replace(metadata_tmp, metadata_path)
    return frame

def validate_bank(bank):
    # Independent validation policies; this is a diagnostic, not model selection.
    rows = []
    for entry in bank:
        if entry.split != "validation": continue
        other = Scripted(1-entry.role, "safe")
        for ep in range(CFG.validation_episodes):
            result = rollout(entry.agent, other, entry.role, .1,
                             stable_seed("bank_quality", entry.identifier, ep), "validation")
            rows.append({"role": CFG.role_names[entry.role], "family": entry.family,
                         "opponent_seed": entry.seed, **result})
    return pd.DataFrame(rows)

BANK, BANK_COST, RESULTS, TRAIN_LOGS, COSTS, SAMPLER_LOGS = [], pd.DataFrame(), pd.DataFrame(), [], [], []
if RUN_EXPERIMENTS:
    BANK, BANK_COST = build_bank()
    BANK_COST.to_csv(OUT / "bank_cost.csv", index=False)
    quality = validate_bank(BANK)
    quality.to_csv(OUT / "bank_validation.csv", index=False)
    print("Validation-only bank diagnostic (not a test leaderboard):")
    display(quality.groupby(["role", "family"])["win"].mean().unstack().round(3))
    bank_meta = [{"id": e.identifier, "split": e.split, "family": e.family, "role": e.role,
                  "seed": e.seed, "checkpoint": e.checkpoint,
                  "parameter_sha256": parameter_digest(e.agent)} for e in BANK]
    (OUT / "bank_manifest.json").write_text(json.dumps(bank_meta, indent=2))
    frames = []
    for role in (0, 1):
        for seed in CFG.seeds:
            for condition in CONDITIONS:
                print(f"{CFG.role_names[role]} | seed={seed} | {condition.name}", flush=True)
                training = train_condition(BANK, role, seed, condition)
                frames.append(evaluate_condition(BANK, role, seed, condition, training))
                common = {"condition": condition.name, "arm": condition.arm,
                          "role": CFG.role_names[role], "seed": seed}
                TRAIN_LOGS.extend([{**common, **r} for r in training["logs"]])
                SAMPLER_LOGS.extend([{**common, **r} for r in training["sampler"].history])
                COSTS.append({**common, "training_steps": training["step"],
                              "gradient_updates": training["agent"].updates,
                              "validation_steps": training["validation_steps"],
                              "training_seconds": training["seconds"],
                              "final_loss": np.mean(training["agent"].losses[-20:])})
    RESULTS = pd.concat(frames, ignore_index=True)
    RESULTS.to_csv(OUT / "all_evaluation_episodes.csv", index=False)
    pd.DataFrame(TRAIN_LOGS).to_csv(OUT / "training_episodes.csv", index=False)
    pd.DataFrame(COSTS).to_csv(OUT / "learner_cost.csv", index=False)
    if SAMPLER_LOGS:
        pd.DataFrame(SAMPLER_LOGS).to_csv(OUT / "adaptive_probabilities.csv", index=False)
    print("Evaluation episodes:", len(RESULTS), "| LP failures:", LP_FAILURES)
    print("Outputs:", OUT.resolve())
else:
    print("Definitions and checks ready. Set RUN_EXPERIMENTS=True, then Run all.")

## 8. Ανάλυση χωρίς ψευδο-επαναλήψεις

Πρώτα μέσος όρος επεισοδίων ανά policy × p_fail, μετά ίσο βάρος στις πολιτικές
μιας οικογένειας και στις οικογένειες. Οι ανεξάρτητες μονάδες για τα 95% CI
είναι **training seeds**, όχι χιλιάδες επεισόδια του ίδιου μοντέλου.
Τα CI είναι t-intervals μεταξύ seeds, υπό το συγκεκριμένο, κοινό opponent bank.
Δεν καλύπτουν πλήρως αβεβαιότητα από νέους πληθυσμούς αντιπάλων· για αυτό
χρειάζονται και επαναλήψεις κατασκευής ολόκληρου του bank σε επόμενη μελέτη.

Προκαθορισμένο κύριο endpoint: macro win rate σε **unseen families και
p_fail > 0.15**. Δευτερεύοντα: όλες οι άγνωστες πολιτικές, curves ανά p_fail,
worst cell στο προκαθορισμένο πλέγμα, hazards και χρόνος σύλληψης υπό capture.
Το worst-cell είναι θορυβώδης εκτίμηση, όχι απόδειξη πραγματικού minimax robustness.
Οι αρνητικές/μηδενικές διαφορές διατηρούνται. Δεν ανακηρύσσεται αυτόματα νικητής.

Το B−A εκτιμά diversity με σταθερό p. Το D−C εκτιμά diversity με randomized p.
Η αλληλεπίδραση είναι `(D−C)−(B−A)`. A/C controls πρώτα μέσος όρος ανά seed.
Οι training win curves έχουν διαφορετικούς αντιπάλους και είναι μόνο διαγνωστικές.

In [ ]:
def seed_interval(values, bounded=False):
    x = np.asarray(values, dtype=float)
    n, mean = len(x), float(np.mean(x))
    half = float(student_t.ppf(.975, n-1)*np.std(x, ddof=1)/np.sqrt(n)) if n > 1 else np.nan
    lo, hi = mean-half, mean+half
    if bounded and n > 1:
        lo, hi = max(0.0, lo), min(1.0, hi)
    return {"mean": mean, "ci_low": lo, "ci_high": hi, "training_seeds": n}

def summarize_seeds(frame, keys, value="win", bounded=True):
    rows = []
    for key, group in frame.groupby(keys, dropna=False):
        if not isinstance(key, tuple): key = (key,)
        rows.append({**dict(zip(keys, key)), **seed_interval(group[value], bounded)})
    return pd.DataFrame(rows)

if not RESULTS.empty:
    policy_keys = ["condition", "arm", "seed", "role", "group", "family", "p_fail", "opponent"]
    POLICY_CELLS = RESULTS.groupby(policy_keys, as_index=False).agg(
        win=("win", "mean"), hazard_rate=("own_hazard", "mean"),
        capture_steps=("capture_steps", "mean"), steps=("steps", "mean"))
    FAMILY_CELLS = POLICY_CELLS.groupby(policy_keys[:-1], as_index=False)[
        ["win", "hazard_rate", "capture_steps", "steps"]].mean()
    SCORES = FAMILY_CELLS.groupby(["condition", "arm", "seed", "role", "group"], as_index=False)["win"].mean()
    SUMMARY = summarize_seeds(SCORES, ["condition", "arm", "role", "group"])
    SUMMARY.to_csv(OUT / "summary_seed_CI.csv", index=False)
    print("Mean win rates — " + ("SMOKE, NOT RESEARCH EVIDENCE" if PROFILE == "smoke" else PROFILE))
    display(SUMMARY.round(3))

    # Primary endpoint: unknown families at noise levels beyond training range.
    primary_cells = FAMILY_CELLS[(FAMILY_CELLS.group == "unseen_family") &
                                (FAMILY_CELLS.p_fail > max(CFG.train_p))]
    PRIMARY_SEEDS = primary_cells.groupby(["condition", "arm", "role", "seed"], as_index=False)["win"].mean()
    PRIMARY = summarize_seeds(PRIMARY_SEEDS, ["condition", "arm", "role"])
    PRIMARY.to_csv(OUT / "primary_endpoint.csv", index=False)
    display(PRIMARY.round(3))

    # Average fixed controls within each paired training seed, not across episodes.
    arm_seed = PRIMARY_SEEDS.groupby(["arm", "role", "seed"], as_index=False)["win"].mean()
    pivot = arm_seed.pivot(index=["role", "seed"], columns="arm", values="win")
    contrasts = {"diversity_at_fixed_p_B-A": pivot.B-pivot.A,
                 "diversity_at_random_p_D-C": pivot.D-pivot.C,
                 "randomization_single_C-A": pivot.C-pivot.A,
                 "randomization_pool_D-B": pivot.D-pivot.B,
                 "combined_D-A": pivot.D-pivot.A,
                 "interaction_D-C-B+A": pivot.D-pivot.C-pivot.B+pivot.A}
    if "E" in pivot:
        contrasts["adaptive_E-D"] = pivot.E-pivot.D
    contrast_frames = [series.rename("difference").reset_index().assign(contrast=name)
                       for name, series in contrasts.items()]
    CONTRAST_SEEDS = pd.concat(contrast_frames, ignore_index=True)
    CONTRASTS = summarize_seeds(CONTRAST_SEEDS, ["role", "contrast"], "difference", False)
    CONTRASTS.to_csv(OUT / "paired_primary_contrasts.csv", index=False)
    print("Paired differences in primary win rate (0.10 = 10 percentage points):")
    display(CONTRASTS.round(3))

    worst = FAMILY_CELLS[FAMILY_CELLS.group != "seen_policy"].groupby(
        ["condition", "role", "seed"], as_index=False)["win"].min()
    WORST = summarize_seeds(worst, ["condition", "role"])
    WORST.to_csv(OUT / "worst_observed_family_p_cell.csv", index=False)
    # Descriptive seen/unseen-policy gap restricted to common known families.
    gap = SCORES.pivot(index=["condition", "role", "seed"], columns="group", values="win")
    gap["seen_minus_unseen_policy"] = gap.seen_policy-gap.unseen_policy
    gap.reset_index().to_csv(OUT / "descriptive_generalization_gap.csv", index=False)
    terminal_mix = RESULTS.groupby(["condition", "role", "group", "reason"], as_index=False).size()
    terminal_mix["fraction"] = terminal_mix["size"]/terminal_mix.groupby(
        ["condition", "role", "group"])["size"].transform("sum")
    terminal_mix.to_csv(OUT / "terminal_reasons_episode_weighted.csv", index=False)

## 9. Γραφήματα και ερμηνεία

Οι σκιασμένες περιοχές δείχνουν 95% CI μεταξύ training seeds. Η γκρίζα περιοχή
επισημαίνει υψηλότερο noise από το training range. Για A/C, οι curves είναι
μέσοι όροι των προκαθορισμένων fixed controls. Τα raw condition-level CSV
παραμένουν διαθέσιμα. Στο smoke αναμένουμε μεγάλες διακυμάνσεις και ελλιπή μάθηση.
Αύξηση του p_fail μπορεί να βοηθά τον έναν ρόλο· δεν επιβάλλουμε μονοτονία.

In [ ]:
if not RESULTS.empty:
    curve = FAMILY_CELLS[FAMILY_CELLS.group != "seen_policy"].groupby(
        ["condition", "arm", "role", "seed", "p_fail"], as_index=False)["win"].mean()
    curve = curve.groupby(["arm", "role", "seed", "p_fail"], as_index=False)["win"].mean()
    curve_summary = summarize_seeds(curve, ["arm", "role", "p_fail"])
    colors = {"A": "#718096", "B": "#2563eb", "C": "#d97706", "D": "#059669", "E": "#9333ea"}
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
    for ax, role in zip(axes, CFG.role_names):
        for arm, g in curve_summary[curve_summary.role == role].groupby("arm"):
            g = g.sort_values("p_fail")
            ax.plot(g.p_fail, g["mean"], marker="o", label=arm, color=colors[arm])
            ax.fill_between(g.p_fail, g.ci_low, g.ci_high, alpha=.10, color=colors[arm])
        ax.axvspan(max(CFG.train_p)+.001, max(CFG.test_p)+.005, color="gray", alpha=.08)
        ax.set(title=role.title()+" — unseen opponents", xlabel="Action failure probability", ylim=(0, 1))
        ax.legend(title="Arm", ncol=5, loc="lower center")
    axes[0].set_ylabel("Macro win rate")
    fig.suptitle("Opponent diversity × stochasticity" + (" | SMOKE ONLY" if PROFILE == "smoke" else ""))
    fig.tight_layout()
    fig.savefig(OUT / "robustness_curves.png", bbox_inches="tight")
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
    for ax, role in zip(axes, CFG.role_names):
        sub = arm_seed[arm_seed.role == role]
        summary = summarize_seeds(sub, ["arm"])
        for i, row in enumerate(summary.itertuples()):
            ax.bar(i, row.mean, color=colors[row.arm], alpha=.8)
            ax.errorbar(i, row.mean, yerr=[[row.mean-row.ci_low], [row.ci_high-row.mean]],
                        fmt="none", capsize=4, color="#222222")
            values = sub[sub.arm == row.arm]["win"].to_numpy()
            ax.scatter(i+np.linspace(-.1, .1, len(values)), values, s=20, color="#111111", zorder=3)
        ax.set_xticks(range(len(summary)), summary.arm)
        ax.set(title=role.title(), xlabel="Arm", ylim=(0, 1))
    axes[0].set_ylabel("Primary endpoint win rate")
    fig.suptitle("Unknown families & p_fail > 0.15" + (" | SMOKE ONLY" if PROFILE == "smoke" else ""))
    fig.tight_layout()
    fig.savefig(OUT / "primary_endpoint.png", bbox_inches="tight")
    plt.show()

    display(pd.DataFrame(COSTS).groupby("condition").agg(
        mean_training_seconds=("training_seconds", "mean"),
        real_training_steps=("training_steps", "mean"),
        extra_validation_steps=("validation_steps", "mean"),
        gradient_updates=("gradient_updates", "mean")).round(2))
    print("Shared bootstrap cost:", int(BANK_COST.steps.sum()), "environment steps;",
          round(BANK_COST.seconds.sum(), 1), "seconds.")
    if SAMPLER_LOGS:
        sample = pd.DataFrame(SAMPLER_LOGS)
        last = sample[(sample.role == "chaser") & (sample.seed == CFG.seeds[0])]
        last = last[last.step == last.step.max()]
        table = last.pivot(index="family", columns="p_fail", values="probability")
        print("Example adaptive probabilities — validation-driven, not test-driven:")
        display(table.round(3))

## 10. Τι συμπέρασμα επιτρέπεται να βγάλουμε;

1. **Πρώτα έλεγχος ποιότητας μάθησης.** Τα smoke/pilot αποτελέσματα και αδύναμοι
   tabular αντίπαλοι δεν τεκμηριώνουν πλεονέκτημα μεθόδου. Ελέγξτε το bank validation,
   το training budget και τη σταθερότητα ανά seed πριν επενδύσετε σε full runs.
2. Θετικό B−A μόνο στο γνωστό περιβάλλον δεν αποδεικνύει robustness στη στοχαστικότητα.
   Εξετάστε D−C, D−B και το προκαθορισμένο κύριο endpoint.
3. Αν το CI περιλαμβάνει το μηδέν, η συγκεκριμένη δοκιμή δεν δίνει σαφή ένδειξη
   διαφοράς. Αυτό δεν αποδεικνύει ισοδυναμία. Οι πολλές δευτερεύουσες συγκρίσεις
   είναι εξερευνητικές· για confirmatory claims απαιτείται προεγγραφή/διόρθωση πολλαπλότητας.
4. Το E πρέπει να συγκριθεί με D και ως προς πρόσθετα validation steps/χρόνο.
   Η απλή σύγκριση ίδιων training steps δεν σημαίνει ίδιο συνολικό compute.
5. Μεταφορά σε νέους χάρτες και άγνωστο, μη παρατηρούμενο p_fail δεν αξιολογούνται
   εδώ. Ο χάρτης παραμένει σταθερός και το p_fail είναι γνωστό input.
6. Αυτό είναι comparative study population training + dynamics randomization.
   Δεν ισχυρίζεται νέα μέθοδο PSRO, Nash-Q υλοποίηση ή αποδεδειγμένη πρωτοτυπία.

**Πλήρης χρήση:** επιλέξτε `PROFILE="full"`, ελέγξτε το printed budget, και Run all.
Για leave-one-family-out τρέξτε ξεχωριστά `HOLDOUT_FAMILY="fp"`, `"minimax"`,
`"dynaq"`, `"dqn"`. Κάθε φορά ανακατασκευάζεται το ελεγχόμενο protocol namespace.
Για outcome-only ablation αλλάξτε `REWARD_MODE="outcome"` σε όλα τα arms μαζί.
Η αλλαγή profile απαιτεί Run all από την αρχή, ώστε οι globals/agents να συμφωνούν.

## Αναφορές

* Lanctot et al. (2017), [A Unified Game-Theoretic Approach to Multiagent RL](https://arxiv.org/abs/1711.00832).
* Pardo et al. (2018), [Time Limits in Reinforcement Learning](https://proceedings.mlr.press/v80/pardo18a.html).
* Lu et al. (2011), [Policy Invariance under Reward Transformations for General-Sum Stochastic Games](https://arxiv.org/abs/1401.3907).
* van Hasselt et al. (2016), [Deep Reinforcement Learning with Double Q-learning](https://arxiv.org/abs/1509.06461).
* Wang et al. (2016), [Dueling Network Architectures for Deep Reinforcement Learning](https://proceedings.mlr.press/v48/wangf16.html).

Οι αριθμοί που παράγονται από το notebook είναι αποτελέσματα της επιλεγμένης
εκτέλεσης και δεν αντιγράφονται από τις αναφορές ή από τα reports του αρχικού ZIP.